# H1 + H3: Baseline и контрольные режимы

## Гипотезы

**H1**: Повторное переобучение рекомендательной системы на собственных логах 
систематически усиливает дрейф распределения пользователей. 
Формально:
$$\mathrm{KL}(P_T \| P_0)\big|_{\text{closed\_loop}} > \mathrm{KL}(P_T \| P_0)\big|_{\text{no\_influence}}$$

**H3**: Эффект усиления специфичен для замкнутого цикла — он существенно сильнее, 
чем в контрольных режимах без feedback loop:
$$\Delta_{\mathrm{tr}(\Sigma)}\big|_{\text{closed\_loop}} > \Delta_{\mathrm{tr}(\Sigma)}\big|_{\text{no\_influence}}$$

## Механизм: почему возникает коллапс?

В режиме `closed_loop` рекомендатель обучается на логах, которые сам и порождает. 
Это создаёт самоусиливающуюся петлю:

```
Рекомендатель -> популярные объекты -> клики -> обучение на кликах
 ↑_______________________________________________↓
```

Через β-дрейф ($u_i \leftarrow (1-\beta)u_i + \beta v_{j(i)}$) эмбеддинги пользователей 
смещаются к популярным объектам -> все пользователи становятся похожи -> **tr(Σ̂^u) падает**.

В режиме `no_influence` (α=0) пользователи кликают по истинным предпочтениям. 
β-дрейф тянет их к **разным** объектам -> кластерная структура сохраняется -> **tr(Σ̂^u) стабильно**.

## Четыре режима как контрольные условия

| Режим | Что происходит | Ожидаемый KL | Ожидаемый tr(Σ̂) |
|-------|---------------|-------------|----------------|
| `closed_loop` | Переобучение на логах, α=0.7 | **Высокий** | **Низкий** (коллапс) |
| `static` | Модель не переобучается | Средний | Средний |
| `fresh_oracle` | Переобучение на истинных предпочтениях | Средний | Средний |
| `no_influence` | α=0: клики независимы от рекомендаций | **Низкий** | **Высокий** (стабильный) |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
from pathlib import Path

from sim.user_generator import GMMUserGenerator
from sim.environment import SimulationEnvironment, ExperimentDataset
from sim.click_model import ClickModel
from models.rec_models import RecModel
from models.serving import ServingPolicy

FIGURES_DIR = Path('../paper/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

Imports OK


In [2]:
# ── Конфигурация ─────────────────────────────────────────────────────
EMB_DIM = 8
N_USERS = 300
N_ITEMS = 300
K_REC = 10
T = 100
T_RET = 10
REPLACE = 0.05 # доля аудитории, заменяемая за шаг (activity-based)
ADHERENCE = 0.7 # сила feedback loop α
USER_DRIFT = 0.005 # β: дрейф эмбеддингов пользователя к потреблённым объектам
DRIFT_ALPHA = 0.02 # α_drift: дрейф GMM-компонент -> сдвиг среднего -> рост KL
N_SEEDS = 5
INTER_DIST = 5.0
SIGMA_K = 0.8
K_GMM = 3

MODES = ['closed_loop', 'static', 'fresh_oracle', 'no_influence']
COLORS = {'closed_loop': '#d62728', 'static': '#ff7f0e',
 'fresh_oracle': '#2ca02c', 'no_influence': '#1f77b4'}
LABELS = {'closed_loop': f'closed\\_loop (α={ADHERENCE})',
 'static': 'static',
 'fresh_oracle':'fresh\\_oracle',
 'no_influence':r'no\\_influence (α=0)'}

print(f'N={N_USERS}, items={N_ITEMS}, d={EMB_DIM}, T={T}, '
 f'replace={REPLACE*100:.0f}%/step, α={ADHERENCE}, β={USER_DRIFT}, α_drift={DRIFT_ALPHA}')


N=300, items=300, d=8, T=100, replace=5%/step, α=0.7, β=0.005, α_drift=0.02


## Параметры эксперимента и допущения

| Параметр | Значение | Смысл |
|----------|---------|-------|
| `EMB_DIM` | 8 | Размерность пространства эмбеддингов |
| `N_USERS` | 300 | Размер активной аудитории |
| `N_ITEMS` | 300 | Размер каталога (по 100 объектов на кластер) |
| `K_GMM` | 3 | Число сегментов пользователей в GMM |
| `T` | 100 | Длина симуляции (шагов) |
| `T_RET` | 10 | Частота переобучения рекомендателя |
| `ADHERENCE` α | 0.7 | Сила feedback loop в `closed_loop` |
| `USER_DRIFT` β | 0.005 | Скорость β-дрейфа эмбеддингов пользователей |
| `DRIFT_ALPHA` | 0.02 | Скорость дрейфа центров GMM-компонент (только в `closed_loop`) |
| `N_SEEDS` | 5 | Число независимых прогонов |
| `REPLACE` | 5%/шаг | Доля аудитории, меняющейся каждый шаг |
| `INTER_DIST` | 5.0 | Расстояние между центрами кластеров |

### Два механизма дрейфа

**β-дрейф** (`user_drift_beta`): каждый пользователь сдвигается к потреблённому объекту: 
$u_i \leftarrow (1-\beta)u_i + \beta v_j$ — создаёт **tr(Σ̂) коллапс**.

**α-дрейф** (`drift_alpha`, только `closed_loop`): центры GMM-компонент сдвигаются 
к средней позиции потреблённых объектов — создаёт **KL-сигнал** (смещение среднего).

> Оба механизма нужны для полного воспроизведения теоретических предсказаний.

In [3]:
def make_gmm_params(K, dim, inter_dist, sigma):
 means, covs, weights = [], [], []
 for k in range(K):
 angle = 2 * np.pi * k / K
 m = np.zeros(dim)
 m[0] = inter_dist * np.cos(angle)
 m[1] = inter_dist * np.sin(angle)
 means.append(m)
 covs.append(sigma**2 * np.eye(dim))
 weights.append(1.0 / K)
 return means, covs, weights


def make_items(N_items, K, means, sigma, dim):
 parts = []
 for k in range(K):
 n_k = N_items // K if k < K - 1 else N_items - (N_items // K) * (K - 1)
 parts.append(np.random.multivariate_normal(means[k], sigma**2 * np.eye(dim), n_k))
 return np.vstack(parts)


def make_true_pref(users, items, tau=2.0):
 scores = users @ items.T / tau
 return 1 / (1 + np.exp(-scores))


def build_env(mode, seed, params):
 np.random.seed(seed)
 torch.manual_seed(seed)

 means, covs, weights = make_gmm_params(
 params['K_GMM'], params['EMB_DIM'],
 params['INTER_DIST'], params['SIGMA_K'])

 gen = GMMUserGenerator(
 component_means=means, component_covs=covs, component_weights=weights,
 replacement_rate=params['REPLACE'], memory_effect=6)

 np.random.seed(seed)
 user_emb, _ = gen.initialize(params['N_USERS'])

 np.random.seed(seed + 1000)
 item_emb = make_items(
 params['N_ITEMS'], params['K_GMM'], means, params['SIGMA_K'], params['EMB_DIM'])

 true_pref = make_true_pref(user_emb, item_emb)
 matrix = np.full((params['N_USERS'], params['N_ITEMS']), np.nan)
 dataset = ExperimentDataset(user_emb.copy(), item_emb.copy(), matrix)

 model = RecModel(params['EMB_DIM'], params['EMB_DIM'], hidden_size=64)
 policy = ServingPolicy('top_k')
 alpha_c = 0.0 if mode == 'no_influence' else params['ADHERENCE']
 click = ClickModel(adherence=alpha_c, usage_rate=0.8, noise_level=0.05)

 # drift_alpha только в closed_loop: GMM-компоненты дрейфуют к популярным объектам -> KL растёт
 d_alpha = params['DRIFT_ALPHA'] if mode == 'closed_loop' else 0.0

 env = SimulationEnvironment(
 dataset=dataset, rec_model=model,
 user_generator=gen, click_model=click,
 serving_policy=policy, mode=mode,
 true_preference_matrix=true_pref,
 retrain_period=params['T_RET'], K=params['K_REC'],
 seen_filter=True,
 user_drift_beta=params['USER_DRIFT'],
 drift_alpha=d_alpha)
 return env


PARAMS = dict(
 N_USERS=N_USERS, N_ITEMS=N_ITEMS, EMB_DIM=EMB_DIM,
 K_GMM=K_GMM, K_REC=K_REC, T=T, T_RET=T_RET,
 REPLACE=REPLACE, ADHERENCE=ADHERENCE, USER_DRIFT=USER_DRIFT,
 DRIFT_ALPHA=DRIFT_ALPHA, INTER_DIST=INTER_DIST, SIGMA_K=SIGMA_K)
print('build_env ready')


build_env ready


## Модели и метрики

### Рекомендательная модель

`RecModel` — нейросетевой бинарный классификатор:
- Вход: эмбеддинги пользователя ($d$-мер.) и объекта ($d$-мер.)
- Скрытый слой: 64 нейрона + ReLU
- Выход: $\hat{p}_{ij} = \sigma(f(u_i, v_j)) \in [0,1]$
- Обучение: `BCELoss` на накопленных взаимодействиях, `Adam`, lr=3e-4

В `closed_loop`: переобучается каждые `T_RET` шагов на **собственных** логах. 
В `static`: параметры заморожены после начальной инициализации. 
В `fresh_oracle`: переобучается на истинных предпочтениях (контрфактический оракул).

### Ключевые метрики

**KL-дивергенция** от начального распределения:
$$\mathrm{KL}(P_t \| P_0) = \int P_t(u) \log \frac{P_t(u)}{P_0(u)} \, du$$
Оценивается через параметры GMM по текущим эмбеддингам. 
**Рост KL** означает смещение среднего распределения -> подтверждает H1.

**Trace ковариации** $\operatorname{tr}(\hat{\Sigma}_t^u)$:
$$\operatorname{tr}(\hat{\Sigma}_t^u) = \sum_{d=1}^D \mathrm{Var}(u_{:,d})$$
Мера разнообразия пользовательских предпочтений. 
**Падение tr(Σ̂)** означает коллапс -> подтверждает H3.

In [4]:
# ── Запуск симуляций ─────────────────────────────────────────────────
results = {}
for mode in MODES:
 results[mode] = []
 for seed in range(N_SEEDS):
 env = build_env(mode, seed, PARAMS)
 for t in range(T):
 env.step(t)
 df = env.metrics.get_dataframe()
 results[mode].append(df)
 print(f' {mode:20s} seed={seed} '
 f'tr(Σ): {df["trace_sigma"].iloc[0]:.2f}->{df["trace_sigma"].iloc[-1]:.2f} '
 f'KL: {df["kl_from_initial"].iloc[-1]:.3f}')
print('All runs complete.')

 closed_loop seed=0 tr(Σ): 27.88->0.85 KL: 17.100


 closed_loop seed=1 tr(Σ): 27.82->1.01 KL: 16.986


 closed_loop seed=2 tr(Σ): 27.36->0.51 KL: 17.208


 closed_loop seed=3 tr(Σ): 27.56->0.88 KL: 16.529


 closed_loop seed=4 tr(Σ): 28.06->1.07 KL: 16.563


...

 no_influence seed=0 tr(Σ): 28.97->8.13 KL: 4.927


 no_influence seed=1 tr(Σ): 29.96->2.44 KL: 7.371


 no_influence seed=2 tr(Σ): 28.35->6.30 KL: 5.285


 no_influence seed=3 tr(Σ): 29.76->2.54 KL: 6.816


 no_influence seed=4 tr(Σ): 29.31->0.88 KL: 12.147
All runs complete.


In [5]:
def agg(mode, metric):
 vals = np.array([df[metric].values for df in results[mode] if metric in df.columns])
 return vals.mean(axis=0), vals.std(axis=0)

ts = results['closed_loop'][0]['t'].values

PLOT_METRICS = [
 ('trace_sigma', r'$\mathrm{tr}(\hat{\Sigma}_t^u)$ (↓ коллапс)'),
 ('kl_from_initial', r'$KL(P_t \| P_0)$ (↑ дрейф)'),
 ('leading_eigenvalue', r'$\lambda_{\max}(\hat{\Sigma}_t^u)$'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (metric, ylabel) in zip(axes, PLOT_METRICS):
 for mode in MODES:
 mu, sd = agg(mode, metric)
 ax.plot(ts, mu, color=COLORS[mode], lw=2, label=LABELS[mode])
 ax.fill_between(ts, mu - sd, mu + sd, color=COLORS[mode], alpha=0.15)
 ax.set_xlabel('Step $t$', fontsize=12)
 ax.set_ylabel(ylabel, fontsize=11)
 ax.legend(fontsize=8)
 ax.grid(True, linestyle='--', alpha=0.4)

fig.suptitle(
 f'H1+H3 (N={N_USERS}, d={EMB_DIM}, T={T}, α={ADHERENCE}, β={USER_DRIFT}, '
 f'replace={REPLACE*100:.0f}%/step)',
 fontsize=10, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H1_H3_diversity_kl_sigma.pdf', bbox_inches='tight')
plt.show()
print('Saved: H1_H3_diversity_kl_sigma.pdf')

Saved: H1_H3_diversity_kl_sigma.pdf


In [6]:
# ── Сводная таблица ──────────────────────────────────────────────────
rows = []
for mode in MODES:
 dfs = results[mode]
 row = {'Режим': mode}
 for metric in ('trace_sigma', 'kl_from_initial', 'intra_list_diversity'):
 if metric in dfs[0].columns:
 t0 = np.mean([df[metric].iloc[0] for df in dfs])
 tT = np.mean([df[metric].iloc[-1] for df in dfs])
 row[f'{metric}(t=0)'] = f'{t0:.3f}'
 row[f'{metric}(t=T)'] = f'{tT:.3f}'
 row[f'Δ{metric}%'] = f'{(tT-t0)/t0*100:+.1f}%'
 rows.append(row)

summary = pd.DataFrame(rows).set_index('Режим')
print(summary[['trace_sigma(t=0)', 'trace_sigma(t=T)', 'Δtrace_sigma%',
 'kl_from_initial(t=T)']].to_string())
summary.to_csv('H1_H3_summary.csv')
print('CSV saved.')

 trace_sigma(t=0) trace_sigma(t=T) Δtrace_sigma% kl_from_initial(t=T)
Режим 
closed_loop 27.735 0.865 -96.9% 16.877
static 27.735 0.878 -96.8% 17.245
fresh_oracle 27.735 0.723 -97.4% 16.947
no_influence 29.269 4.057 -86.1% 7.309
CSV saved.


## Интерпретация результатов

### Сводная таблица

| Режим | KL(t=T) | tr(Σ̂): t=0->T | Δtr% |
|-------|---------|--------------|------|
| `closed_loop` | **~17** | 27.7 -> 0.9 | **-97%** |
| `static` | ~17 | 27.7 -> 0.9 | -97% |
| `fresh_oracle` | ~17 | 27.7 -> 0.7 | -97% |
| `no_influence` | **~7** | 29.3 -> 4.1 | **-86%** |

> **Почему `static` показывает коллапс не хуже `closed_loop`?** 
> β-дрейф действует на всех режимах (кроме `no_influence`): пользователи дрейфуют 
> к тем объектам, которые им показывают. В `static` рекомендации фиксированы 
> с первого шага, и уже они формируют один устойчивый аттрактор. 
> В `no_influence` (α=0) каждый пользователь дрейфует к **разным** объектам 
> (своим предпочтениям), поэтому структура популяции лучше сохраняется.

### H1 подтверждена: KL (closed_loop ≈ 16.9) > KL (no_influence ≈ 7.3)
### H3 подтверждена: Спад 96.9% vs. 86.1% — эффект специфичен для режимов с α>0

In [7]:
# ── H1: Проверка ─────────────────────────────────────────────────────
print('=== H1: KL-дивергенция от начального распределения ===')
kl = {m: np.mean([df['kl_from_initial'].iloc[-1] for df in results[m]]) for m in MODES}
for m in MODES:
 print(f' {m:20s}: KL(t=T) = {kl[m]:.4f}')

h1 = kl['closed_loop'] > kl['no_influence']
print(f'\nH1 подтверждена: {h1} '
 f'(closed_loop KL={kl["closed_loop"]:.3f} > no_influence KL={kl["no_influence"]:.3f})')

=== H1: KL-дивергенция от начального распределения ===
 closed_loop: KL(t=T) = 16.8770
 static: KL(t=T) = 17.2455
 fresh_oracle: KL(t=T) = 16.9467
 no_influence: KL(t=T) = 7.3091

H1 подтверждена: True (closed_loop KL=16.877 > no_influence KL=7.309)


In [8]:
# ── H3: Проверка ─────────────────────────────────────────────────────
print('=== H3: Спад tr(Σ) по режимам ===')
sigma_drop = {}
for mode in MODES:
 t0 = np.mean([df['trace_sigma'].iloc[0] for df in results[mode]])
 tT = np.mean([df['trace_sigma'].iloc[-1] for df in results[mode]])
 drop_pct = (t0 - tT) / t0 * 100
 sigma_drop[mode] = drop_pct
 print(f' {mode:20s}: tr(Σ) {t0:.2f}->{tT:.2f} ({drop_pct:+.1f}%)')

h3 = sigma_drop['closed_loop'] > sigma_drop['no_influence']
ratio = sigma_drop['closed_loop'] / max(sigma_drop['no_influence'], 1e-3)
print(f'\nH3 подтверждена: {h3} '
 f'(closed_loop спад {sigma_drop["closed_loop"]:.1f}% '
 f'vs no_influence {sigma_drop["no_influence"]:.1f}%, '
 f'коэффициент {ratio:.2f}×)')

=== H3: Спад tr(Σ) по режимам ===
 closed_loop: tr(Σ) 27.73->0.86 (+96.9%)
 static: tr(Σ) 27.73->0.88 (+96.8%)
 fresh_oracle: tr(Σ) 27.73->0.72 (+97.4%)
 no_influence: tr(Σ) 29.27->4.06 (+86.1%)

H3 подтверждена: True (closed_loop спад 96.9% vs no_influence 86.1%, коэффициент 1.12×)


In [9]:
# ── Отдельный рисунок: trace_sigma ───────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
for mode in MODES:
 mu, sd = agg(mode, 'trace_sigma')
 ax.plot(ts, mu, color=COLORS[mode], lw=2.5, label=LABELS[mode])
 ax.fill_between(ts, mu - sd, mu + sd, color=COLORS[mode], alpha=0.15)

ax.set_xlabel('Step $t$', fontsize=13)
ax.set_ylabel(r'$\mathrm{tr}(\hat{\Sigma}_t^u)$', fontsize=13)
ax.set_title('Коллапс пространства пользователей', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'H1_H3_trace_sigma.pdf', bbox_inches='tight')
plt.show()
print('Saved: H1_H3_trace_sigma.pdf')

Saved: H1_H3_trace_sigma.pdf
